# Артём Юдин М4121

In [3]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import median_absolute_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Считывание и предобработка данных, используя шаги из прошлой лабораторной

In [4]:
X_train = pd.read_csv("../data/preprocessed/X_train_smart_regex_7020samples.csv")
X_val = pd.read_csv("../data/preprocessed/X_val_smart_regex_1980samples.csv")
X_test = pd.read_csv("../data/preprocessed/X_test_smart_regex_1000samples.csv")

y_train = pd.read_csv("../data/preprocessed/y_train_7020samples.csv")
y_val = pd.read_csv("../data/preprocessed/y_val_1980samples.csv")
y_test = pd.read_csv("../data/preprocessed/y_test_1000samples.csv")

# объединение токенов вместе
for df in (X_train, X_val, X_test):
    df["untokenized_ttext"] = df["ttext"].apply(lambda x: " ".join(eval(x)))

df = 5e-4
# инициализация векторизаторов
bow_vectorizer = CountVectorizer(min_df=df, max_df=1 - df)
tdidf_vectorizer = TfidfVectorizer(min_df=df, max_df=1 - df)

# инициализируем скейлеры
bow_scaler = StandardScaler()
tfidf_scaler = StandardScaler()

# масштабируем данные
X_train_bow = bow_scaler.fit_transform(bow_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray())
X_val_bow = bow_scaler.transform(bow_vectorizer.transform(X_val["untokenized_ttext"]).toarray())
X_test_bow = bow_scaler.transform(bow_vectorizer.transform(X_test["untokenized_ttext"]).toarray())

X_train_tfidf = tfidf_scaler.fit_transform(tdidf_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray())
X_val_tfidf = tfidf_scaler.transform(tdidf_vectorizer.transform(X_val["untokenized_ttext"]).toarray())
X_test_tfidf = tfidf_scaler.transform(tdidf_vectorizer.transform(X_test["untokenized_ttext"]).toarray())

# Функция для обучения

$$MSE = \frac{\sum_{i=1}^{n}(y_i-\hat{y_i})^2}{n}$$
$$MAE = \frac{\sum_{i=1}^{n}|y_i-\hat{y_i}|}{n}$$
$$Median AE = median(|y_i-\hat{y_i}|)$$
$$R^2 = 1 - \frac{D[\hat{y}]}{D[y]}$$

In [ ]:
def train(X_train_bow, X_train_tfidf, y_train, X_val_bow, X_val_tfidf, y_val) -> dict[str, list]:
    metrics = {
        "MSE": mean_squared_error,
        "MAE": mean_absolute_error,
        "R^2": r2_score,
        "Median Absolute Error": median_absolute_error,
    }

    models = {
        "Linear Regression": LinearRegression(),
        "L1 Regression": Lasso(alpha=1000, random_state=42),
        "L2 Regression": Ridge(alpha=1000, random_state=42),
        "Elastic": ElasticNet(alpha=1000, l1_ratio=0.5, random_state=42),
    }
    df_dict = {
        "Model": [],
        "Encoding": [],
        "MSE": [],
        "MAE": [],
        "R^2": [],
        "Median Absolute Error": []
    }

    for model_name, model in models.items():
        df_dict["Model"].append(model_name)
        df_dict["Encoding"].append("BoW")

        # обучаем модель
        model.fit(X_train_bow, y_train)
        y_pred = model.predict(X_val_bow)

        # рассчитываем метрики для обученного набора
        for metric_name, metric in metrics.items():
            df_dict[metric_name].append(round(metric(y_val, y_pred), 2))

        df_dict["Model"].append(model_name)
        df_dict["Encoding"].append("TF-IDF")

        # обучаем модель
        model.fit(X_train_tfidf, y_train)
        y_pred = model.predict(X_val_tfidf)

        # рассчитываем метрики для обученного набора
        for metric_name, metric in metrics.items():
            df_dict[metric_name].append(round(metric(y_val, y_pred), 2))
    
    return df_dict


In [6]:
df = pd.DataFrame(
    train(X_train_bow, X_train_tfidf, y_train, X_val_bow, X_val_tfidf, y_val)
)
df

,Model,Encoding,MSE,MAE,R^2,Median Absolute Error
0,Linear Regression,BoW,1.02,0.86,-0.02,0.82
1,Linear Regression,TF-IDF,1.02,0.85,-0.02,0.79
2,L1 Regression,BoW,1.00,1.00,0.00,1.00
3,L1 Regression,TF-IDF,1.00,1.00,0.00,1.00
4,L2 Regression,BoW,0.93,0.84,0.07,0.82
5,L2 Regression,TF-IDF,0.94,0.83,0.06,0.81
6,Elastic,BoW,1.00,1.00,0.00,1.00
7,Elastic,TF-IDF,1.00,1.00,0.00,1.00


# Проведём отбор признаков

In [ ]:
# отбор признаков
bow_pca = PCA(1360, random_state=42)
tfidf_pca = PCA(1786, random_state=42)

X_train_bow_pca = bow_pca.fit_transform(X_train_bow)
X_val_bow_pca = bow_pca.transform(X_val_bow)
X_test_bow_pca = bow_pca.transform(X_test_bow)

X_train_tfidf_pca = tfidf_pca.fit_transform(X_train_tfidf)
X_val_tfidf_pca = tfidf_pca.transform(X_val_tfidf)
X_test_tfidf_pca = tfidf_pca.transform(X_test_tfidf)

# обучение модели
df_pca = pd.DataFrame(train(X_train_bow_pca, X_train_tfidf_pca, y_train, X_val_bow_pca, X_val_tfidf_pca, y_val))
df_pca

,Model,Encoding,MSE,MAE,R^2,Median Absolute Error
0,Linear Regression,BoW,0.90,0.83,0.10,0.82
1,Linear Regression,TF-IDF,0.98,0.84,0.02,0.81
2,L1 Regression,BoW,1.00,1.00,0.00,1.00
3,L1 Regression,TF-IDF,1.00,1.00,0.00,1.00
4,L2 Regression,BoW,0.88,0.83,0.12,0.83
5,L2 Regression,TF-IDF,0.93,0.83,0.07,0.82
6,Elastic,BoW,1.00,1.00,0.00,1.00
7,Elastic,TF-IDF,1.00,1.00,0.00,1.00


# Предсказание длины

In [8]:
y_train = X_train["tlen"]
y_val   = X_val["tlen"]

df_tlen = pd.DataFrame(
    train(X_train_bow, X_train_tfidf, y_train, X_val_bow, X_val_tfidf, y_val)
)
df_tlen

,Model,Encoding,MSE,MAE,R^2,Median Absolute Error
0,Linear Regression,BoW,542.88,17.75,0.30,13.96
1,Linear Regression,TF-IDF,642.41,19.71,0.17,15.98
2,L1 Regression,BoW,771.33,23.30,-0.00,21.76
3,L1 Regression,TF-IDF,771.33,23.30,-0.00,21.76
4,L2 Regression,BoW,499.32,17.17,0.35,13.72
5,L2 Regression,TF-IDF,643.81,20.01,0.17,16.56
6,Elastic,BoW,771.33,23.30,-0.00,21.76
7,Elastic,TF-IDF,771.33,23.30,-0.00,21.76


In [9]:
df_pca_tlen = pd.DataFrame(train(X_train_bow_pca, X_train_tfidf_pca, y_train, X_val_bow_pca, X_val_tfidf_pca, y_val))
df_pca_tlen

,Model,Encoding,MSE,MAE,R^2,Median Absolute Error
0,Linear Regression,BoW,532.57,17.83,0.31,14.37
1,Linear Regression,TF-IDF,877.46,23.44,-0.14,19.41
2,L1 Regression,BoW,771.33,23.30,-0.00,21.76
3,L1 Regression,TF-IDF,771.33,23.30,-0.00,21.76
4,L2 Regression,BoW,523.12,17.86,0.32,14.60
5,L2 Regression,TF-IDF,821.47,22.93,-0.07,19.54
6,Elastic,BoW,771.33,23.30,-0.00,21.76
7,Elastic,TF-IDF,771.33,23.30,-0.00,21.76


# Эксп с классификацией (можно не смотреть)

In [29]:
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression()
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)
lr.fit(X_train_bow_pca, y_train_enc)
y_pred = lr.predict(X_val_bow_pca)
print(classification_report(y_val_enc, y_pred))

/home/artyom/myprojects/ITMO/IIML/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/artyom/myprojects/ITMO/IIML/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


              precision    recall  f1-score   support

           0       0.65      0.67      0.66       990
           1       0.66      0.63      0.64       990

    accuracy                           0.65      1980
   macro avg       0.65      0.65      0.65      1980
weighted avg       0.65      0.65      0.65      1980



In [ ]:
X_train_bow = bow_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray()
X_val_bow = bow_vectorizer.transform(X_val["untokenized_ttext"]).toarray()

lr = LogisticRegression()
lr.fit(X_train_bow, y_train_enc)
y_pred = lr.predict(X_val_bow)
print(classification_report(y_val_enc, y_pred))

              precision    recall  f1-score   support

           0       0.66      0.69      0.67       990
           1       0.67      0.65      0.66       990

    accuracy                           0.67      1980
   macro avg       0.67      0.67      0.67      1980
weighted avg       0.67      0.67      0.67      1980



# Эксп с отсутствием масштабирования (можно не смотреть)

In [32]:
X_train_tfidf = tdidf_vectorizer.fit_transform(X_train["untokenized_ttext"]).toarray()
X_val_tfidf = tdidf_vectorizer.transform(X_val["untokenized_ttext"]).toarray()

pd.DataFrame(
    train(X_train_bow, X_train_tfidf, y_train, X_val_bow, X_val_tfidf, y_val)
)

,Model,Encoding,MSE,MAE,R^2,Median Absolute Error
0,Linear Regression,BoW,1.02,0.86,-0.02,0.82
1,Linear Regression,TF-IDF,1.02,0.85,-0.02,0.79
2,L1 Regression,BoW,1.00,1.00,0.00,1.00
3,L1 Regression,TF-IDF,1.00,1.00,0.00,1.00
4,L2 Regression,BoW,0.93,0.96,0.07,0.98
5,L2 Regression,TF-IDF,0.99,0.99,0.01,1.00
6,Elastic,BoW,1.00,1.00,0.00,1.00
7,Elastic,TF-IDF,1.00,1.00,0.00,1.00


# Выводы